# Normalization of the dataset
Calculate the values required for a Z Normalization of the images and heigth profiles are determined and stored in a JSON file for later usage in the model

In [1]:
from pathlib import Path
import os
import json

import numpy as np
import torch
import torchvision.io
from torchvision.transforms import v2

In [2]:
DATA_PATH = Path("../../data/train")
BEFORE_PATH = "before"
AFTER_PATH = "after"
IMAGE_PATH = "image"
HEIGHT_PATH = "height"
MASK_PATH = "mask"

In [3]:
scale = v2.ToDtype(torch.float32, scale=True)

In [4]:
def get_image_as_numpy(image_path, mask):
    img = scale(torchvision.io.read_image(image_path)).detach().numpy()
    return np.ma.array(img, mask=mask)

def get_height_as_numpy(height_path, mask):
    return np.ma.array(np.load(height_path), mask=mask)

def get_mask_as_numpy(mask_path, data_type=None):
    "Load mask"
    mask = np.load(mask_path)
    if data_type=="image":
        mask = np.repeat(mask[np.newaxis, :, :], 3, axis=0)
    return mask

def calc_image_stats(img):
    "Calculate number of elements, sum of elements and sum of elements squared per channel"
    return (img.count(axis=(1, 2)), img.sum(axis=(1, 2)), (img*img).sum(axis=(1, 2)))

def calc_height_stats(height):
    "Calculate number of elements, sum of elements and sum of elements squared"
    return (np.array([height.count()]), np.array([height.sum()]), np.array([(height*height).sum()]))

def get_sample_ids(file_path):
    return [i[:-4] for i in os.listdir(file_path)]

def calc_std(count, sum, sum_squared):
    return np.sqrt(count * sum_squared - sum**2)/count

def calc_mean(count, sum):
    return sum/count

def calc_statistics(data_path, time, data_type):
    sample_ids = get_sample_ids(data_path / time / data_type)
    total_count = 0
    total_sum = 0
    total_sum_sqrd = 0

    for id in sample_ids:
        mask = get_mask_as_numpy(data_path / MASK_PATH / f"{id}.npy", data_type=data_type)

        if data_type == "image":
            image = get_image_as_numpy(data_path / time / data_type / f"{id}.png", mask)
            count, sum, sum_squared = calc_image_stats(image)
        elif data_type == "height":
            height = get_height_as_numpy(data_path / time / data_type / f"{id}.npy", mask)
            count, sum, sum_squared = calc_height_stats(height)
        else:
            raise ValueError

        total_count += count
        total_sum += sum
        total_sum_sqrd += sum_squared

    return len(sample_ids), calc_mean(total_count, total_sum), calc_std(total_count, total_sum, total_sum_sqrd)

In [5]:
results = {}

for time in [BEFORE_PATH, AFTER_PATH]:
    data_dict = {}
    for data_type in [IMAGE_PATH, HEIGHT_PATH]:
        # Calculate statistics across all files of the time and data_type
        n, mean, std = calc_statistics(DATA_PATH, time, data_type)
        print(f"The {time} {data_type} statistics for {n} files are mean: {mean} and std: {std}")
        data_dict[data_type] = {"mean": list(mean), "std": list(std)}
    results[time] = data_dict

# Write statistics to json file
with open(DATA_PATH / "normalization.json", mode="w") as write_file:
    json.dump(results, write_file)

The before image statistics for 642 files are mean: [0.43928119685876793 0.4151601222521586 0.3885703104794666] and std: [0.19328407201866618 0.183195305833475 0.1749009466912351]
The before height statistics for 642 files are mean: [0.00323275] and std: [0.01286774]
The after image statistics for 642 files are mean: [0.5164524916412389 0.49247321009075373 0.4664985848103354] and std: [0.16287072644941739 0.16025532424491293 0.16448512182032024]
The after height statistics for 642 files are mean: [-0.00411228] and std: [0.01318253]


In [6]:
def check_statistics(time, data_type):
    """Check if calculating the standard z score of every single data set leads to mean zero and std 1"""
    if data_type == "image":
        file_path = DATA_PATH / time / IMAGE_PATH
    elif data_type == "height":
        file_path = DATA_PATH / time / HEIGHT_PATH
    else:
        raise ValueError

    # Load the statistics from json
    with open(DATA_PATH / "normalization.json", mode="r") as write_file:
        results = json.load(write_file)


    sample_ids = get_sample_ids(file_path)
    total_count = 0
    total_sum = 0
    total_sum_sqrd = 0

    for id in sample_ids:
        mask = get_mask_as_numpy(DATA_PATH / MASK_PATH / f"{id}.npy", data_type=data_type)

        if data_type == "image":
            image = get_image_as_numpy(DATA_PATH / time / IMAGE_PATH / f"{id}.png", mask)
            image = (image - np.array(results[time][data_type]["mean"])[:, np.newaxis, np.newaxis]) / np.array(results[time][data_type]["std"])[:, np.newaxis, np.newaxis]
            count, sum, sum_squared = calc_image_stats(image)
        elif data_type == "height":
            height = get_height_as_numpy(DATA_PATH / time / HEIGHT_PATH / f"{id}.npy", mask)
            height = (height - np.array(results[time][data_type]["mean"])) / np.array(results[time][data_type]["std"])
            count, sum, sum_squared = calc_height_stats(height)
        else:
            raise ValueError

        total_count += count
        total_sum += sum
        total_sum_sqrd += sum_squared

    return len(sample_ids), calc_mean(total_count, total_sum), calc_std(total_count, total_sum, total_sum_sqrd)

In [7]:
for time in [BEFORE_PATH, AFTER_PATH]:
    data_dict = {}
    for data_type in ["image", "height"]:
        # Calculate statistics across all files of the time and data_type
        n, mean, std = check_statistics(time, data_type)
        print(f"The {time} {data_type} statistics for {n} files are mean: {mean} and std: {std}")
        data_dict[data_type] = {"mean": list(mean), "std": list(std)}

The before image statistics for 642 files are mean: [-0.0011915024330263263 -0.0016897919509252502 -0.002258880186740467] and std: [1.0030898338221899 1.0042260411655664 1.0054020122256866]
The before height statistics for 642 files are mean: [1.4716268e-16] and std: [1.]
The after image statistics for 642 files are mean: [0.0001777043094816198 -0.00022446936183544128 -0.0007474096531293826] and std: [0.9993875378366645 1.0008397988394468 1.0023804835556116]
The after height statistics for 642 files are mean: [1.1435107e-19] and std: [1.]
